In [ ]:
# 06.0 策略研究启动、规则冻结与输入审计
# 每次重启实验室后，先运行本单元。

from pathlib import Path
import importlib
import sys
import numpy as np
import pandas as pd
PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
LIB_DIR = PROJECT_ROOT / "lib"

if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import profitability_research_utils as pru
pru = importlib.reload(pru)

PATHS = pru.ProjectPaths.from_root(PROJECT_ROOT)
PATHS.ensure_directories()
PATHS.ensure_cache_ignored()
import pandas as pd
import rqdatac
import strategy_core as sc

import matplotlib.pyplot as plt
sc = importlib.reload(sc)
import ricequant_data_utils as rqutils
rqutils = importlib.reload(rqutils)

# -------------------------------------------------------------------
# 固定策略规则：后续回测不得依据表现修改这些设定
# -------------------------------------------------------------------
STRATEGY_SPEC = {
    "strategy_name": (
        "integrated_profitability_structure_proxy_q5_long_only"
    ),
    "factor_source": (
        "05_integrated_profitability_structure_proxy.pkl"
    ),
    "factor_definition": (
        "基于研报四个底层因子的等权透明实现：稳定去杠杆 ROE、稳定 ROIC 代理、"
        "稳定 RONOA 代理、重构 FCFFIC 各 25%；组合后行业和对数市值中性化；不是研报原始两层权重公式。"
    ),
    "strategy_scope": (
        "低覆盖率等权透明实现的执行可行性检验，"
        "不是事后挑选 FCFFIC 或盈利稳定性的新组合。"
    ),
    "universe": "动态中证 800",
    "benchmark_index": "000906.XSHG",
    "portfolio_rule": "每期选择因子最高 20% 股票，等权做多",
    "signal_frequency": "月度信号",
    "execution_rule": "信号日后首个交易日开盘调仓",
    "holding_rule": "持有 20 个交易日，按预先保存的收益标签时间表执行",
    "short_leg": (
        "同时报告理论 Q5-Q1 作为因子诊断；"
        "不将其称为可直接执行的 A 股做空策略。"
    ),
    "initial_cash": 100000000,
    "cash_interest_yield_annual": 0.02,
    "teacher_cost_assumption": {
        "buy_cost": 0.0003,
        "sell_cost": 0.0008,
        "minimum_fee": 5,
        "explanation": (
            "对应老师框架：买入过户费等加佣金 0.03%，"
            "卖出额外包含 0.05% 印花税，总计 0.08%。"
        ),
    },
    "out_of_sample_period": "2023-01-01 至 2025-12-31",
    "anti_overfitting_rule": (
        "不得根据回测结果修改组件、权重、分组阈值、"
        "交易成本或样本外时间范围。"
    ),
}

required_inputs = {
    "综合因子本地缓存": (
        PATHS.cache_data_dir
        / "05_integrated_profitability_structure_proxy.pkl"
    ),
    "中性化上下文本地缓存": (
        PATHS.cache_data_dir
        / "05_neutralization_context.pkl"
    ),
    "20 日收益标签本地缓存": (
        PATHS.cache_data_dir
        / "05_forward_return_20d.pkl"
    ),
    "收益标签时间表": (
        PATHS.table_dir
        / "05_return_schedule.csv"
    ),
    "综合因子有效性汇总": (
        PATHS.table_dir
        / "05_integrated_profitability_effectiveness_summary.csv"
    ),
}

missing_inputs = [
    name for name, path in required_inputs.items()
    if not path.exists()
]

if missing_inputs:
    raise RuntimeError(
        "策略回测缺少输入：" + "、".join(missing_inputs)
        + "。请先不要回测，先补齐对应缓存或公开结果。"
    )

specification_path = (
    PATHS.metadata_dir
    / "06_integrated_profitability_strategy_specification.json"
)

pru.save_public_json(
    STRATEGY_SPEC,
    specification_path,
)

print("===== 06 策略研究已启动 =====")
print("策略名称：", STRATEGY_SPEC["strategy_name"])
print("股票池：", STRATEGY_SPEC["universe"])
print("组合规则：", STRATEGY_SPEC["portfolio_rule"])
print("执行规则：", STRATEGY_SPEC["execution_rule"])
print("持有规则：", STRATEGY_SPEC["holding_rule"])
print("基准：", STRATEGY_SPEC["benchmark_index"])
print("交易成本：买入 0.03%，卖出 0.08%，最低 5 元")
print("已冻结策略规则：", specification_path)
print()
print("下一步：06.1 检查 20 日持有窗口是否重叠、是否存在空档，")
print("再决定权重面板和逐日开盘价回测的实现方式。")

In [ ]:
%%writefile /home/rice/notebook/a-share-profitability-factor-replication/lib/strategy_core.py
"""
通用策略研究核心函数。

只存放可复用的策略计算工具：
- 交易日程审计；
- 后续会加入权重构建、通用回测和绩效计算。

不包含任何特定因子、文件名、股票池或研究结论。
"""

from __future__ import annotations

from typing import Any

import numpy as np
import pandas as pd


STRATEGY_CORE_VERSION = "0.1.0"


def audit_execution_schedule(
    schedule: pd.DataFrame,
    signal_column: str,
    entry_column: str,
    exit_column: str,
    trading_calendar: pd.DatetimeIndex | None = None,
    expected_holding_intervals: int | None = None,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    审计任意策略的信号、入场、退出日程。

    参数：
    - schedule：包含信号日、入场日、退出日的 DataFrame；
    - signal_column / entry_column / exit_column：对应列名；
    - trading_calendar：可选交易日历，用于精确计算交易日间隔；
    - expected_holding_intervals：可选预期持有交易日间隔。

    返回：
    - 带有持有期、相邻期衔接关系的日期级审计表；
    - 汇总字典。
    """
    required_columns = {
        signal_column,
        entry_column,
        exit_column,
    }

    missing_columns = required_columns - set(schedule.columns)

    if missing_columns:
        raise ValueError(
            "交易日程缺少列：" + ", ".join(sorted(missing_columns))
        )

    audited = schedule.copy()

    for column in [signal_column, entry_column, exit_column]:
        audited[column] = pd.to_datetime(
            audited[column],
            errors="coerce",
        )

    if audited[[signal_column, entry_column, exit_column]].isna().any().any():
        raise ValueError("交易日程存在无法解析的日期。")

    audited = audited.sort_values(signal_column).reset_index(drop=True)

    audited["entry_before_or_equal_signal"] = (
        audited[entry_column] <= audited[signal_column]
    )
    audited["exit_before_or_equal_entry"] = (
        audited[exit_column] <= audited[entry_column]
    )

    audited["next_entry_date"] = audited[entry_column].shift(-1)
    audited["calendar_days_next_entry_minus_exit"] = (
        audited["next_entry_date"] - audited[exit_column]
    ).dt.days

    if trading_calendar is not None:
        calendar = pd.DatetimeIndex(
            pd.to_datetime(trading_calendar)
        ).sort_values().unique()

        calendar_position = pd.Series(
            np.arange(len(calendar)),
            index=calendar,
        )

        audited["entry_trading_position"] = audited[entry_column].map(
            calendar_position
        )
        audited["exit_trading_position"] = audited[exit_column].map(
            calendar_position
        )
        audited["next_entry_trading_position"] = audited[
            "next_entry_date"
        ].map(calendar_position)

        audited["holding_trading_intervals"] = (
            audited["exit_trading_position"]
            - audited["entry_trading_position"]
        )

        audited["next_entry_minus_exit_trading_intervals"] = (
            audited["next_entry_trading_position"]
            - audited["exit_trading_position"]
        )

        audited["calendar_date_not_found"] = (
            audited["entry_trading_position"].isna()
            | audited["exit_trading_position"].isna()
        )

        audited["transition_type"] = np.select(
            [
                audited["next_entry_date"].isna(),
                audited["next_entry_minus_exit_trading_intervals"] < 0,
                audited["next_entry_minus_exit_trading_intervals"] == 0,
                audited["next_entry_minus_exit_trading_intervals"] > 0,
            ],
            [
                "last_period",
                "overlap",
                "same_open_rebalance",
                "cash_gap",
            ],
            default="unknown",
        )
    else:
        audited["holding_trading_intervals"] = np.nan
        audited["next_entry_minus_exit_trading_intervals"] = np.nan
        audited["calendar_date_not_found"] = False

        audited["transition_type"] = np.select(
            [
                audited["next_entry_date"].isna(),
                audited["next_entry_date"] < audited[exit_column],
                audited["next_entry_date"] == audited[exit_column],
                audited["next_entry_date"] > audited[exit_column],
            ],
            [
                "last_period",
                "overlap",
                "same_open_rebalance",
                "cash_gap",
            ],
            default="unknown",
        )

    valid_holding = audited["holding_trading_intervals"].dropna()

    summary = {
        "signal_period_count": int(len(audited)),
        "first_signal_date": (
            audited[signal_column].min().strftime("%Y-%m-%d")
            if not audited.empty
            else None
        ),
        "last_signal_date": (
            audited[signal_column].max().strftime("%Y-%m-%d")
            if not audited.empty
            else None
        ),
        "entry_before_or_equal_signal_count": int(
            audited["entry_before_or_equal_signal"].sum()
        ),
        "exit_before_or_equal_entry_count": int(
            audited["exit_before_or_equal_entry"].sum()
        ),
        "overlap_period_count": int(
            (audited["transition_type"] == "overlap").sum()
        ),
        "same_open_rebalance_count": int(
            (audited["transition_type"] == "same_open_rebalance").sum()
        ),
        "cash_gap_period_count": int(
            (audited["transition_type"] == "cash_gap").sum()
        ),
        "calendar_date_not_found_count": int(
            audited["calendar_date_not_found"].sum()
        ),
        "mean_holding_trading_intervals": (
            float(valid_holding.mean())
            if not valid_holding.empty
            else np.nan
        ),
        "min_holding_trading_intervals": (
            int(valid_holding.min())
            if not valid_holding.empty
            else np.nan
        ),
        "max_holding_trading_intervals": (
            int(valid_holding.max())
            if not valid_holding.empty
            else np.nan
        ),
    }

    if expected_holding_intervals is not None:
        audited["matches_expected_holding_intervals"] = (
            audited["holding_trading_intervals"]
            == expected_holding_intervals
        )

        comparable_periods = audited[
            "holding_trading_intervals"
        ].notna()

        summary["expected_holding_intervals"] = int(
            expected_holding_intervals
        )
        summary["holding_interval_mismatch_count"] = int(
            (
                comparable_periods
                & ~audited["matches_expected_holding_intervals"]
            ).sum()
        )

    return audited, summary


In [ ]:
# 06.1 审计冻结收益标签的执行时间表
# 目的：
# 1. 确认 T+1 开盘买入、T+21 开盘卖出的日期映射；
# 2. 检查连续信号之间是否重叠持仓或存在现金空档；
# 3. 据此确定 06.2 应采用普通连续调仓还是重叠持仓分组回测。
#
# 本单元不重新下载数据、不重新计算因子、不重建收益标签。

required_session_variables = ["PROJECT_ROOT", "pd", "rqdatac", "sc"]

missing_session_variables = [
    name for name in required_session_variables
    if name not in globals()
]

if missing_session_variables:
    raise RuntimeError(
        "请先重新运行 06.0 项目启动单元。缺少变量："
        + ", ".join(missing_session_variables)
    )

# ----------------------------------------------------------------
# 1. 读取 05 中已经冻结的收益标签时间表
# ----------------------------------------------------------------
SCHEDULE_PATH = PROJECT_ROOT / "artifacts" / "tables" / "05_return_schedule.csv"
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"

SCHEDULE_SIGNAL_COL = "signal_date"
SCHEDULE_ENTRY_COL = "entry_date_t_plus_1_open"
SCHEDULE_EXIT_COL = "exit_date_t_plus_21_open"
SCHEDULE_HOLDING_COL = "holding_intervals"
SCHEDULE_STATUS_COL = "status"

if not SCHEDULE_PATH.exists():
    raise FileNotFoundError(
        f"找不到已冻结的收益标签时间表：{SCHEDULE_PATH}\n"
        "请先运行 05 中生成 05_return_schedule.csv 的单元。"
    )

return_schedule = pd.read_csv(SCHEDULE_PATH)

required_schedule_columns = {
    SCHEDULE_SIGNAL_COL,
    SCHEDULE_ENTRY_COL,
    SCHEDULE_EXIT_COL,
    SCHEDULE_HOLDING_COL,
    SCHEDULE_STATUS_COL,
}

missing_schedule_columns = required_schedule_columns - set(return_schedule.columns)

if missing_schedule_columns:
    raise RuntimeError(
        "05_return_schedule.csv 的字段与预期不一致。"
        "缺少：" + ", ".join(sorted(missing_schedule_columns))
        + "；实际字段为：" + ", ".join(return_schedule.columns)
    )

for column in [
    SCHEDULE_SIGNAL_COL,
    SCHEDULE_ENTRY_COL,
    SCHEDULE_EXIT_COL,
]:
    return_schedule[column] = pd.to_datetime(return_schedule[column])

valid_schedule = return_schedule.loc[
    return_schedule[SCHEDULE_STATUS_COL].astype(str).str.lower().eq("ok")
].copy()

if valid_schedule.empty:
    raise RuntimeError(
        "收益标签时间表中没有 status='ok' 的有效记录。\n"
        "实际 status 取值为："
        + ", ".join(
            sorted(return_schedule[SCHEDULE_STATUS_COL].astype(str).unique())
        )
    )

valid_schedule = valid_schedule.sort_values(SCHEDULE_SIGNAL_COL).reset_index(drop=True)

# ----------------------------------------------------------------
# 2. 先核对 05 已保存的持有期标签
# ----------------------------------------------------------------
source_holding_interval_audit = (
    valid_schedule[SCHEDULE_HOLDING_COL]
    .value_counts(dropna=False)
    .rename_axis("holding_intervals")
    .reset_index(name="signal_count")
    .sort_values("holding_intervals")
)

unexpected_holding_rows = source_holding_interval_audit.loc[
    source_holding_interval_audit["holding_intervals"].ne(20)
]

if not unexpected_holding_rows.empty:
    raise RuntimeError(
        "05 收益标签中发现非 20 个交易日的持有期，请先检查：\n"
        + unexpected_holding_rows.to_string(index=False)
    )

# ----------------------------------------------------------------
# 3. 使用真实交易日历审计窗口是否重叠、是否有空档
# ----------------------------------------------------------------
calendar_start = valid_schedule[SCHEDULE_ENTRY_COL].min().strftime("%Y-%m-%d")
calendar_end = valid_schedule[SCHEDULE_EXIT_COL].max().strftime("%Y-%m-%d")

trading_calendar = pd.DatetimeIndex(
    rqdatac.get_trading_dates(calendar_start, calendar_end)
)

schedule_audit, schedule_summary = sc.audit_execution_schedule(
    schedule=valid_schedule,
    signal_column=SCHEDULE_SIGNAL_COL,
    entry_column=SCHEDULE_ENTRY_COL,
    exit_column=SCHEDULE_EXIT_COL,
    trading_calendar=trading_calendar,
    expected_holding_intervals=20,
)

# ----------------------------------------------------------------
# 4. 保存安全审计结果：仅日期与计数，不包含个股数据
# ----------------------------------------------------------------
if isinstance(schedule_summary, pd.DataFrame):
    schedule_summary_table = schedule_summary.copy()
elif isinstance(schedule_summary, pd.Series):
    schedule_summary_table = (
        schedule_summary.rename("value")
        .rename_axis("metric")
        .reset_index()
    )
else:
    schedule_summary_table = pd.DataFrame([schedule_summary])

audit_path = TABLE_DIR / "06_integrated_profitability_execution_schedule_audit.csv"
summary_path = TABLE_DIR / "06_integrated_profitability_execution_schedule_summary.csv"
holding_path = TABLE_DIR / "06_integrated_profitability_source_holding_intervals.csv"

schedule_audit.to_csv(audit_path, index=False, encoding="utf-8-sig")
schedule_summary_table.to_csv(summary_path, index=False, encoding="utf-8-sig")
source_holding_interval_audit.to_csv(holding_path, index=False, encoding="utf-8-sig")

# ----------------------------------------------------------------
# 5. 输出结论：决定后续回测框架
# ----------------------------------------------------------------
print("===== 06 执行时间表审计摘要 =====")
print(schedule_summary_table.to_string(index=False))

print("\n===== 05 已冻结持有期核对 =====")
print(source_holding_interval_audit.to_string(index=False))

print("\n已保存：", audit_path)
print("已保存：", summary_path)
print("已保存：", holding_path)

summary_text = schedule_summary_table.to_string(index=False).lower()

if "overlap" in summary_text and "0" not in summary_text:
    print(
        "\n下一步：存在持有窗口重叠，06.2 将采用“重叠持仓分组（sleeves）”的逐日开盘价回测。"
    )
else:
    print(
        "\n下一步：请把上面的审计摘要发给我；我会据真实的重叠/空档结果确定 06.2 的回测结构。"
    )

In [ ]:
# 06.2 生成冻结综合盈利因子的 Q5 等权目标权重
# 不下载数据；仅读取本地综合因子缓存。
# 含股票代码的权重只写入 cache/，不会导出至 artifacts/。

sc = importlib.reload(sc)

required_session_variables = [
    "PROJECT_ROOT",
    "pd",
    "sc",
    "valid_schedule",
]

missing_session_variables = [
    name for name in required_session_variables
    if name not in globals()
]

if missing_session_variables:
    raise RuntimeError(
        "请先运行 06.0 和 06.1。缺少变量："
        + ", ".join(missing_session_variables)
    )

FACTOR_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "05_integrated_profitability_structure_proxy.pkl"
)

WEIGHT_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_q5_target_weights.pkl"
)

SELECTION_AUDIT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_q5_selection_audit.csv"
)

STRATEGY_SCHEDULE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_eligible_schedule.csv"
)

ELIGIBLE_SCHEDULE_AUDIT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_eligible_execution_audit.csv"
)

ELIGIBLE_SCHEDULE_SUMMARY_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_eligible_execution_summary.csv"
)

SELECTION_FRACTION = 0.20
MIN_VALID_STOCKS = 100

if not FACTOR_CACHE_PATH.exists():
    raise FileNotFoundError(
        f"找不到综合因子本地缓存：{FACTOR_CACHE_PATH}\n"
        "请先运行 05.13.1。"
    )

factor_cache_payload = pd.read_pickle(FACTOR_CACHE_PATH)

if isinstance(factor_cache_payload, pd.DataFrame):
    integrated_factor_panel = factor_cache_payload

elif isinstance(factor_cache_payload, dict):
    dataframe_items = {
        str(key): value
        for key, value in factor_cache_payload.items()
        if isinstance(value, pd.DataFrame)
    }

    preferred_factor_keys = [
    "final_factor",
    "integrated_profitability_structure_proxy_strict",
    "integrated_profitability_structure_proxy",
    "integrated_factor_panel",
    "final_factor_panel",
    "factor_panel",
    "factor",
    "data",]

    matched_keys = [
        key for key in preferred_factor_keys
        if key in dataframe_items
    ]

    if len(matched_keys) == 1:
        integrated_factor_panel = dataframe_items[matched_keys[0]]

        print(
            "综合因子缓存：字典容器；"
            f"已读取其中的 DataFrame：{matched_keys[0]}"
        )

    elif len(dataframe_items) == 1:
        only_key = next(iter(dataframe_items))
        integrated_factor_panel = dataframe_items[only_key]

        print(
            "综合因子缓存：字典容器；"
            f"已读取其中唯一的 DataFrame：{only_key}"
        )

    else:
        payload_schema = ", ".join(
            f"{key}={type(value).__name__}"
            for key, value in factor_cache_payload.items()
        )

        raise RuntimeError(
            "综合因子缓存中存在多个候选 DataFrame，不能安全地猜测哪一个是最终因子面板。\n"
            "请把下面的缓存结构发给我：\n"
            + payload_schema
        )

else:
    raise RuntimeError(
        "综合因子缓存格式不支持："
        + type(factor_cache_payload).__name__
    )

if integrated_factor_panel.empty:
    raise RuntimeError("读取到的综合因子面板为空。")

integrated_factor_panel.index = pd.to_datetime(
    integrated_factor_panel.index
).normalize()

if WEIGHT_CACHE_PATH.exists() and SELECTION_AUDIT_PATH.exists():
    q5_target_weights = pd.read_pickle(WEIGHT_CACHE_PATH)
    q5_selection_audit = pd.read_csv(SELECTION_AUDIT_PATH)

    print("Q5 权重缓存：命中，直接读取本地缓存。")

else:
    q5_target_weights, q5_selection_audit = (
        sc.build_quantile_target_weights(
            score_panel=integrated_factor_panel,
            schedule=valid_schedule,
            signal_column=SCHEDULE_SIGNAL_COL,
            entry_column=SCHEDULE_ENTRY_COL,
            selection_fraction=SELECTION_FRACTION,
            min_valid_assets=MIN_VALID_STOCKS,
            higher_is_better=True,
        )
    )

    WEIGHT_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    q5_target_weights.to_pickle(WEIGHT_CACHE_PATH)

    print("Q5 权重缓存：未命中，已新建本地缓存。")

q5_selection_audit["signal_date"] = pd.to_datetime(
    q5_selection_audit["signal_date"]
)

q5_selection_audit["entry_date"] = pd.to_datetime(
    q5_selection_audit["entry_date"]
)

q5_selection_audit.to_csv(
    SELECTION_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

eligible_signal_dates = set(
    q5_selection_audit.loc[
        q5_selection_audit["eligible"],
        "signal_date",
    ]
)

strategy_schedule = valid_schedule.loc[
    valid_schedule[SCHEDULE_SIGNAL_COL].isin(eligible_signal_dates)
].copy()

strategy_schedule = strategy_schedule.sort_values(
    SCHEDULE_SIGNAL_COL
).reset_index(drop=True)

strategy_schedule.to_csv(
    STRATEGY_SCHEDULE_PATH,
    index=False,
    encoding="utf-8-sig",
)

eligible_execution_audit, eligible_execution_summary = (
    sc.audit_execution_schedule(
        schedule=strategy_schedule,
        signal_column=SCHEDULE_SIGNAL_COL,
        entry_column=SCHEDULE_ENTRY_COL,
        exit_column=SCHEDULE_EXIT_COL,
        trading_calendar=trading_calendar,
        expected_holding_intervals=20,
    )
)

if isinstance(eligible_execution_summary, pd.DataFrame):
    eligible_execution_summary_table = eligible_execution_summary.copy()
elif isinstance(eligible_execution_summary, pd.Series):
    eligible_execution_summary_table = (
        eligible_execution_summary
        .rename("value")
        .rename_axis("metric")
        .reset_index()
    )
else:
    eligible_execution_summary_table = pd.DataFrame(
        [eligible_execution_summary]
    )

eligible_execution_audit.to_csv(
    ELIGIBLE_SCHEDULE_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

eligible_execution_summary_table.to_csv(
    ELIGIBLE_SCHEDULE_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("\n===== Q5 选股覆盖率 =====")
print(
    q5_selection_audit.groupby(
        ["eligible", "reason"],
        dropna=False,
    ).agg(
        signal_periods=("signal_date", "count"),
        mean_valid_stocks=("valid_score_count", "mean"),
        min_valid_stocks=("valid_score_count", "min"),
        mean_selected_stocks=("selected_asset_count", "mean"),
        min_selected_stocks=("selected_asset_count", "min"),
    ).round(2).to_string()
)

print("\n===== 可交易综合因子时间表审计 =====")
print(eligible_execution_summary_table.to_string(index=False))

print("\n===== 本地缓存与安全产物 =====")
print("本地权重缓存（不导出）：", WEIGHT_CACHE_PATH)
print("安全选股审计：", SELECTION_AUDIT_PATH)
print("安全策略日期表：", STRATEGY_SCHEDULE_PATH)
print("安全执行审计：", ELIGIBLE_SCHEDULE_AUDIT_PATH)
print("安全执行摘要：", ELIGIBLE_SCHEDULE_SUMMARY_PATH)

print(
    "\n下一步：06.3 将根据“可交易综合因子时间表审计”的最大重叠情况，"
    "冻结非杠杆 sleeves 的资金分配规则，再读取逐日开盘价执行回测。"
)

In [ ]:
# 06.3 构造重叠持仓批次的非杠杆事件日目标权重
# 权重缓存含股票代码，仅保存在 cache/。
# artifacts 中只保存持仓批次数、日期和资金配置审计。

required_session_variables = [
    "PROJECT_ROOT",
    "pd",
    "importlib",
    "sc",
    "q5_target_weights",
    "strategy_schedule",
    "trading_calendar",
]

missing_session_variables = [
    name for name in required_session_variables
    if name not in globals()
]

if missing_session_variables:
    raise RuntimeError(
        "请先运行 06.0、06.1、06.2。缺少变量："
        + ", ".join(missing_session_variables)
    )

sc = importlib.reload(sc)

EVENT_WEIGHT_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_q5_sleeve_event_targets.pkl"
)

SLEEVE_AUDIT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_sleeve_allocation_audit.csv"
)

ALLOCATION_RULE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "metadata"
    / "06_integrated_profitability_sleeve_allocation_rule.json"
)

if EVENT_WEIGHT_CACHE_PATH.exists() and SLEEVE_AUDIT_PATH.exists():
    sleeve_event_targets = pd.read_pickle(EVENT_WEIGHT_CACHE_PATH)
    sleeve_audit = pd.read_csv(SLEEVE_AUDIT_PATH)

    print("持仓批次目标权重缓存：命中，直接读取。")

else:
    sleeve_event_targets, sleeve_audit = (
        sc.build_active_sleeve_event_targets(
            sleeve_weights=q5_target_weights,
            schedule=strategy_schedule,
            entry_column=SCHEDULE_ENTRY_COL,
            exit_column=SCHEDULE_EXIT_COL,
            trading_calendar=trading_calendar,
        )
    )

    EVENT_WEIGHT_CACHE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    sleeve_event_targets.to_pickle(EVENT_WEIGHT_CACHE_PATH)

    print("持仓批次目标权重缓存：未命中，已新建。")

sleeve_audit["event_date"] = pd.to_datetime(
    sleeve_audit["event_date"]
)

sleeve_audit.to_csv(
    SLEEVE_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

max_active_sleeves = int(
    sleeve_audit["active_sleeve_count"].max()
)

if max_active_sleeves < 1:
    raise RuntimeError("没有生成有效持仓批次。")

allocation_rule = pd.DataFrame(
    [
        {
            "rule_name": "active_sleeve_aggregation",
            "rule_value": "equal_weight_across_active_sleeves",
        },
        {
            "rule_name": "gross_exposure_when_active",
            "rule_value": 1.0,
        },
        {
            "rule_name": "leverage",
            "rule_value": "not_allowed",
        },
        {
            "rule_name": "cash_when_no_active_sleeve",
            "rule_value": 1.0,
        },
        {
            "rule_name": "maximum_observed_active_sleeves",
            "rule_value": max_active_sleeves,
        },
        {
            "rule_name": "execution_price",
            "rule_value": "open_price_on_event_date",
        },
    ]
)

allocation_rule.to_json(
    ALLOCATION_RULE_PATH,
    orient="records",
    force_ascii=False,
    indent=2,
)

print("\n===== 重叠持仓批次资金配置摘要 =====")
print(
    sleeve_audit[
        [
            "active_sleeve_count",
            "gross_target_weight",
            "cash_target_weight",
        ]
    ]
    .groupby("active_sleeve_count")
    .agg(
        event_dates=("gross_target_weight", "count"),
        mean_gross_target_weight=("gross_target_weight", "mean"),
        mean_cash_target_weight=("cash_target_weight", "mean"),
    )
    .round(4)
    .to_string()
)

print("\n最大同时活跃批次数：", max_active_sleeves)
print("本地事件权重缓存（不导出）：", EVENT_WEIGHT_CACHE_PATH)
print("安全资金配置审计：", SLEEVE_AUDIT_PATH)
print("已冻结配置规则：", ALLOCATION_RULE_PATH)

print(
    "\n下一步：06.4 读取逐日开盘价，并使用冻结的事件日目标权重执行真实的"
    "买入、卖出、手续费、最低佣金、现金利息与基准对比回测。"
)


In [ ]:
# 06.4 获取并缓存策略回测所需的逐日价格数据
# 数据仅保存至 cache/；artifacts 仅保存覆盖率审计，不含个股价格或持仓。

required_session_variables = [
    "PROJECT_ROOT",
    "pd",
    "rqdatac",
    "sleeve_event_targets",
    "strategy_schedule",
    "SCHEDULE_ENTRY_COL",
    "SCHEDULE_EXIT_COL",
]

missing_session_variables = [
    name for name in required_session_variables
    if name not in globals()
]

if missing_session_variables:
    raise RuntimeError(
        "请先运行 06.0 至 06.3。缺少变量："
        + ", ".join(missing_session_variables)
    )

BENCHMARK_CODE = "000906.XSHG"
PRICE_ADJUSTMENT = "pre"
PRICE_FREQUENCY = "1d"

PRICE_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_strategy_daily_prices.pkl"
)

DAILY_COVERAGE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_daily_price_coverage.csv"
)

EVENT_COVERAGE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_event_open_coverage.csv"
)

PRICE_SUMMARY_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_market_data_summary.csv"
)

trade_asset_codes = (
    sleeve_event_targets.columns[
        sleeve_event_targets.ne(0.0).any(axis=0)
    ]
    .astype(str)
    .tolist()
)

if not trade_asset_codes:
    raise RuntimeError("持仓批次目标权重中没有任何股票。")

price_start_date = pd.to_datetime(
    strategy_schedule[SCHEDULE_ENTRY_COL].min()
).strftime("%Y-%m-%d")

price_end_date = pd.to_datetime(
    strategy_schedule[SCHEDULE_EXIT_COL].max()
).strftime("%Y-%m-%d")

price_cache_hit = False

if PRICE_CACHE_PATH.exists():
    cached_price_bundle = pd.read_pickle(PRICE_CACHE_PATH)

    required_cache_keys = {
        "open_prices",
        "close_prices",
        "benchmark_close",
    }

    if (
        isinstance(cached_price_bundle, dict)
        and required_cache_keys.issubset(cached_price_bundle)
    ):
        cached_open = cached_price_bundle["open_prices"]
        cached_close = cached_price_bundle["close_prices"]

        cache_has_assets = set(trade_asset_codes).issubset(
            set(cached_open.columns)
        )

        cache_has_dates = (
            pd.Timestamp(price_start_date) >= pd.to_datetime(cached_open.index).min()
            and pd.Timestamp(price_end_date) <= pd.to_datetime(cached_open.index).max()
        )

        if cache_has_assets and cache_has_dates:
            price_bundle = cached_price_bundle
            price_cache_hit = True

if not price_cache_hit:
    print(
        "逐日价格缓存：未命中，开始从 RiceQuant 读取 "
        f"{len(trade_asset_codes)} 只曾入选股票的开盘价与收盘价。"
    )

    open_prices = rqdatac.get_price(
        trade_asset_codes,
        start_date=price_start_date,
        end_date=price_end_date,
        frequency=PRICE_FREQUENCY,
        fields="open",
        adjust_type=PRICE_ADJUSTMENT,
        skip_suspended=False,
        expect_df=True,
    )

    close_prices = rqdatac.get_price(
        trade_asset_codes,
        start_date=price_start_date,
        end_date=price_end_date,
        frequency=PRICE_FREQUENCY,
        fields="close",
        adjust_type=PRICE_ADJUSTMENT,
        skip_suspended=False,
        expect_df=True,
    )

    benchmark_raw = rqdatac.get_price(
        BENCHMARK_CODE,
        start_date=price_start_date,
        end_date=price_end_date,
        frequency=PRICE_FREQUENCY,
        fields="close",
        adjust_type=PRICE_ADJUSTMENT,
        skip_suspended=False,
        expect_df=True,
    )

    if not isinstance(open_prices, pd.DataFrame):
        raise RuntimeError("RiceQuant 返回的开盘价不是 DataFrame。")

    if not isinstance(close_prices, pd.DataFrame):
        raise RuntimeError("RiceQuant 返回的收盘价不是 DataFrame。")

    if isinstance(benchmark_raw, pd.DataFrame):
        benchmark_close = benchmark_raw.iloc[:, 0].copy()
    else:
        benchmark_close = pd.Series(benchmark_raw).copy()

    open_prices.index = pd.to_datetime(open_prices.index).normalize()
    close_prices.index = pd.to_datetime(close_prices.index).normalize()
    benchmark_close.index = pd.to_datetime(benchmark_close.index).normalize()

    open_prices = open_prices.reindex(columns=trade_asset_codes)
    close_prices = close_prices.reindex(columns=trade_asset_codes)

    price_bundle = {
        "open_prices": open_prices,
        "close_prices": close_prices,
        "benchmark_close": benchmark_close,
        "asset_count": len(trade_asset_codes),
        "benchmark_code": BENCHMARK_CODE,
        "adjust_type": PRICE_ADJUSTMENT,
        "start_date": price_start_date,
        "end_date": price_end_date,
    }

    PRICE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(price_bundle, PRICE_CACHE_PATH)

else:
    print("逐日价格缓存：命中，直接读取本地缓存。")

open_prices = price_bundle["open_prices"].copy()
close_prices = price_bundle["close_prices"].copy()
benchmark_close = price_bundle["benchmark_close"].copy()

open_prices.index = pd.to_datetime(open_prices.index).normalize()
close_prices.index = pd.to_datetime(close_prices.index).normalize()
benchmark_close.index = pd.to_datetime(benchmark_close.index).normalize()

open_prices = open_prices.reindex(columns=trade_asset_codes)
close_prices = close_prices.reindex(columns=trade_asset_codes)

daily_price_coverage = pd.DataFrame(
    {
        "date": open_prices.index,
        "open_valid_stock_count": open_prices.notna().sum(axis=1).to_numpy(),
        "close_valid_stock_count": close_prices.notna().sum(axis=1).to_numpy(),
        "benchmark_close_available": benchmark_close.reindex(
            open_prices.index
        ).notna().to_numpy(),
    }
)

event_targets = sleeve_event_targets.copy()
event_targets.index = pd.to_datetime(event_targets.index).normalize()
event_targets = event_targets.reindex(columns=trade_asset_codes).fillna(0.0)

previous_event_targets = event_targets.shift(1).fillna(0.0)

# 事件日既要有新目标股票的开盘价，也要有前一批持仓的卖出开盘价。
event_trade_required = (
    event_targets.ne(0.0)
    | previous_event_targets.ne(0.0)
)

event_open_prices = open_prices.reindex(
    index=event_targets.index,
    columns=trade_asset_codes,
)

event_open_coverage = pd.DataFrame(
    {
        "event_date": event_targets.index,
        "required_trade_stock_count": event_trade_required.sum(axis=1).to_numpy(),
        "open_price_available_count": (
            event_open_prices.notna() & event_trade_required
        ).sum(axis=1).to_numpy(),
    }
)

event_open_coverage["missing_open_price_count"] = (
    event_open_coverage["required_trade_stock_count"]
    - event_open_coverage["open_price_available_count"]
)

event_open_coverage["all_required_open_prices_available"] = (
    event_open_coverage["missing_open_price_count"].eq(0)
)

market_data_summary = pd.DataFrame(
    [
        {
            "strategy_trade_asset_count": len(trade_asset_codes),
            "price_start_date": price_start_date,
            "price_end_date": price_end_date,
            "trading_day_count": len(open_prices.index),
            "mean_open_valid_stock_count": round(
                daily_price_coverage["open_valid_stock_count"].mean(),
                2,
            ),
            "min_open_valid_stock_count": int(
                daily_price_coverage["open_valid_stock_count"].min()
            ),
            "mean_close_valid_stock_count": round(
                daily_price_coverage["close_valid_stock_count"].mean(),
                2,
            ),
            "min_close_valid_stock_count": int(
                daily_price_coverage["close_valid_stock_count"].min()
            ),
            "event_date_count": len(event_open_coverage),
            "event_dates_with_missing_open_price": int(
                (~event_open_coverage[
                    "all_required_open_prices_available"
                ]).sum()
            ),
            "benchmark_missing_close_days": int(
                (~daily_price_coverage[
                    "benchmark_close_available"
                ]).sum()
            ),
            "price_adjustment": PRICE_ADJUSTMENT,
            "benchmark_code": BENCHMARK_CODE,
        }
    ]
)

daily_price_coverage.to_csv(
    DAILY_COVERAGE_PATH,
    index=False,
    encoding="utf-8-sig",
)

event_open_coverage.to_csv(
    EVENT_COVERAGE_PATH,
    index=False,
    encoding="utf-8-sig",
)

market_data_summary.to_csv(
    PRICE_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("\n===== 策略逐日市场数据摘要 =====")
print(market_data_summary.to_string(index=False))

print("\n===== 事件日开盘价可执行性 =====")
print(
    event_open_coverage[
        "all_required_open_prices_available"
    ].value_counts().rename_axis(
        "all_required_open_prices_available"
    ).reset_index(
        name="event_date_count"
    ).to_string(index=False)
)

print("\n本地价格缓存（不导出）：", PRICE_CACHE_PATH)
print("安全逐日覆盖率审计：", DAILY_COVERAGE_PATH)
print("安全事件开盘价审计：", EVENT_COVERAGE_PATH)
print("安全市场数据摘要：", PRICE_SUMMARY_PATH)

print(
    "\n下一步：06.5 仅使用本地缓存价格，构建通用事件日开盘价回测引擎，"
    "纳入买入 0.03%、卖出 0.08%、最低佣金 5 元和现金年化 2%。"
)

In [ ]:
%%writefile /home/rice/notebook/a-share-profitability-factor-replication/lib/ricequant_data_utils.py
"""通用 RiceQuant 数据结构适配函数。"""

import pandas as pd

RQ_DATA_UTILS_VERSION = "0.1.0"


def to_wide_price_panel(raw_price_data, *, value_column=None):
    """
    将 API 返回的价格数据转换为“日期 × 资产”的宽表。

    支持：
    - 已是 DatetimeIndex 的宽表；
    - (日期, 资产) 或 (资产, 日期) 的 MultiIndex 长表；
    - 索引元素为 tuple 的普通 Index 长表。

    参数：
    - raw_price_data：Series 或 DataFrame
    - value_column：长表中存放价格的列名；若只有一列可省略

    返回：
    - DataFrame，索引为标准化交易日期、列为资产代码
    """
    if isinstance(raw_price_data, pd.Series):
        column_name = value_column or raw_price_data.name or "value"
        frame = raw_price_data.to_frame(name=column_name)

    elif isinstance(raw_price_data, pd.DataFrame):
        frame = raw_price_data.copy()

    else:
        raise TypeError(
            "raw_price_data 必须是 pandas Series 或 DataFrame，"
            f"实际为：{type(raw_price_data).__name__}"
        )

    if (
        not isinstance(frame.index, pd.MultiIndex)
        and len(frame.index) > 0
        and isinstance(frame.index[0], tuple)
    ):
        frame.index = pd.MultiIndex.from_tuples(frame.index)

    if isinstance(frame.index, pd.DatetimeIndex):
        wide_panel = frame.copy()

    elif isinstance(frame.index, pd.MultiIndex):
        if frame.index.nlevels != 2:
            raise ValueError(
                "长表价格索引必须恰好包含“日期、资产”两个层级。"
            )

        if value_column in frame.columns:
            value_series = frame[value_column]

        elif frame.shape[1] == 1:
            value_series = frame.iloc[:, 0]

        else:
            raise ValueError(
                "无法确定价格列。请提供 value_column；"
                "当前列为：" + ", ".join(map(str, frame.columns))
            )

        datetime_levels = []

        for level_number in range(frame.index.nlevels):
            level_values = frame.index.get_level_values(level_number)

            converted_values = pd.to_datetime(
                level_values,
                errors="coerce",
            )

            if pd.Series(converted_values).notna().all():
                datetime_levels.append(level_number)

        if len(datetime_levels) != 1:
            raise ValueError(
                "无法唯一识别 MultiIndex 中的日期层级。"
            )

        date_level = datetime_levels[0]
        asset_level = 1 - date_level

        wide_panel = value_series.unstack(level=asset_level)

    else:
        raise ValueError(
            "无法识别价格数据索引结构："
            + type(frame.index).__name__
        )

    wide_panel.index = pd.DatetimeIndex(
        pd.to_datetime(wide_panel.index)
    ).normalize()

    if wide_panel.index.has_duplicates:
        raise ValueError("转换后的价格面板存在重复日期。")

    wide_panel.columns = wide_panel.columns.astype(str)

    return wide_panel.sort_index()

In [ ]:
# 06.4 修复：复用已下载但尚未缓存的 RiceQuant 原始价格数据
# 此单元只为本次异常恢复而临时导入适配库。
# 后续将 rqutils 的导入放到 06.0 即可。

import importlib
import ricequant_data_utils as rqutils

rqutils = importlib.reload(rqutils)

required_memory_data = [
    "PROJECT_ROOT",
    "pd",
    "open_prices",
    "close_prices",
    "benchmark_raw",
    "sleeve_event_targets",
    "trade_asset_codes",
]

missing_memory_data = [
    name for name in required_memory_data
    if name not in globals()
]

if missing_memory_data:
    raise RuntimeError(
        "本次已下载的价格数据不在当前内存中，不能复用。\n"
        "缺少变量：" + ", ".join(missing_memory_data)
    )

# 将 RiceQuant 的元组索引长表转换为日期 × 股票宽表。
open_prices = rqutils.to_wide_price_panel(
    open_prices,
    value_column="open",
).reindex(columns=trade_asset_codes)

close_prices = rqutils.to_wide_price_panel(
    close_prices,
    value_column="close",
).reindex(columns=trade_asset_codes)

benchmark_panel = rqutils.to_wide_price_panel(
    benchmark_raw,
    value_column="close",
)

benchmark_close = benchmark_panel.iloc[:, 0].copy()

price_bundle = {
    "open_prices": open_prices,
    "close_prices": close_prices,
    "benchmark_close": benchmark_close,
    "asset_count": len(trade_asset_codes),
    "benchmark_code": BENCHMARK_CODE,
    "adjust_type": PRICE_ADJUSTMENT,
    "start_date": price_start_date,
    "end_date": price_end_date,
}

PRICE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.to_pickle(price_bundle, PRICE_CACHE_PATH)

daily_price_coverage = pd.DataFrame(
    {
        "date": open_prices.index,
        "open_valid_stock_count": open_prices.notna().sum(axis=1).to_numpy(),
        "close_valid_stock_count": close_prices.notna().sum(axis=1).to_numpy(),
        "benchmark_close_available": benchmark_close.reindex(
            open_prices.index
        ).notna().to_numpy(),
    }
)

event_targets = sleeve_event_targets.copy()
event_targets.index = pd.to_datetime(event_targets.index).normalize()
event_targets = event_targets.reindex(columns=trade_asset_codes).fillna(0.0)

previous_event_targets = event_targets.shift(1).fillna(0.0)

event_trade_required = (
    event_targets.ne(0.0)
    | previous_event_targets.ne(0.0)
)

event_open_prices = open_prices.reindex(
    index=event_targets.index,
    columns=trade_asset_codes,
)

event_open_coverage = pd.DataFrame(
    {
        "event_date": event_targets.index,
        "required_trade_stock_count": event_trade_required.sum(axis=1).to_numpy(),
        "open_price_available_count": (
            event_open_prices.notna() & event_trade_required
        ).sum(axis=1).to_numpy(),
    }
)

event_open_coverage["missing_open_price_count"] = (
    event_open_coverage["required_trade_stock_count"]
    - event_open_coverage["open_price_available_count"]
)

event_open_coverage["all_required_open_prices_available"] = (
    event_open_coverage["missing_open_price_count"].eq(0)
)

market_data_summary = pd.DataFrame(
    [
        {
            "strategy_trade_asset_count": len(trade_asset_codes),
            "price_start_date": price_start_date,
            "price_end_date": price_end_date,
            "trading_day_count": len(open_prices.index),
            "mean_open_valid_stock_count": round(
                daily_price_coverage["open_valid_stock_count"].mean(),
                2,
            ),
            "min_open_valid_stock_count": int(
                daily_price_coverage["open_valid_stock_count"].min()
            ),
            "mean_close_valid_stock_count": round(
                daily_price_coverage["close_valid_stock_count"].mean(),
                2,
            ),
            "min_close_valid_stock_count": int(
                daily_price_coverage["close_valid_stock_count"].min()
            ),
            "event_date_count": len(event_open_coverage),
            "event_dates_with_missing_open_price": int(
                (~event_open_coverage[
                    "all_required_open_prices_available"
                ]).sum()
            ),
            "benchmark_missing_close_days": int(
                (~daily_price_coverage[
                    "benchmark_close_available"
                ]).sum()
            ),
            "price_adjustment": PRICE_ADJUSTMENT,
            "benchmark_code": BENCHMARK_CODE,
        }
    ]
)

daily_price_coverage.to_csv(
    DAILY_COVERAGE_PATH,
    index=False,
    encoding="utf-8-sig",
)

event_open_coverage.to_csv(
    EVENT_COVERAGE_PATH,
    index=False,
    encoding="utf-8-sig",
)

market_data_summary.to_csv(
    PRICE_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("\n===== 策略逐日市场数据摘要 =====")
print(market_data_summary.to_string(index=False))

print("\n===== 事件日开盘价可执行性 =====")
print(
    event_open_coverage[
        "all_required_open_prices_available"
    ]
    .value_counts()
    .rename_axis("all_required_open_prices_available")
    .reset_index(name="event_date_count")
    .to_string(index=False)
)

print("\n本地价格缓存（不导出）：", PRICE_CACHE_PATH)
print("安全逐日覆盖率审计：", DAILY_COVERAGE_PATH)
print("安全事件开盘价审计：", EVENT_COVERAGE_PATH)
print("安全市场数据摘要：", PRICE_SUMMARY_PATH)

In [ ]:
# 06.5 真实开盘价、成本和现金利息下的策略回测

required_session_variables = [
    "PROJECT_ROOT",
    "pd",
    "np",
    "rqdatac",
    "importlib",
    "sc",
    "rqutils",
]

missing_session_variables = [
    name for name in required_session_variables
    if name not in globals()
]

if missing_session_variables:
    raise RuntimeError(
        "请先运行更新后的 06.0。缺少变量："
        + ", ".join(missing_session_variables)
    )

sc = importlib.reload(sc)

INITIAL_CASH = 100_000_000.0
BUY_COST_RATE = 0.0003
SELL_COST_RATE = 0.0008
MIN_COMMISSION = 5.0
LOT_SIZE = 100
CASH_INTEREST_ANNUAL = 0.02
ANNUALIZATION = 252

EVENT_TARGET_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_q5_sleeve_event_targets.pkl"
)

PRICE_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_strategy_daily_prices.pkl"
)

BACKTEST_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_open_price_backtest.pkl"
)

DAILY_NAV_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_daily_nav.csv"
)

EVENT_TRADE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_event_trade_summary.csv"
)

PERFORMANCE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_performance.csv"
)

EXECUTION_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_execution_summary.csv"
)

if not EVENT_TARGET_CACHE_PATH.exists():
    raise FileNotFoundError(
        "找不到 06.3 的事件日目标权重缓存。"
    )

if not PRICE_CACHE_PATH.exists():
    raise FileNotFoundError(
        "找不到 06.4 的价格缓存。"
    )

sleeve_event_targets = pd.read_pickle(EVENT_TARGET_CACHE_PATH)
price_bundle = pd.read_pickle(PRICE_CACHE_PATH)

required_price_keys = {
    "open_prices",
    "close_prices",
    "benchmark_close",
    "start_date",
    "end_date",
    "benchmark_code",
}

missing_price_keys = required_price_keys - set(price_bundle)

if missing_price_keys:
    raise RuntimeError(
        "价格缓存缺少字段："
        + ", ".join(sorted(missing_price_keys))
    )

# 为使策略与基准都从首个调仓日开盘开始计值，只额外读取基准开盘价。
if "benchmark_open" not in price_bundle:
    benchmark_open_raw = rqdatac.get_price(
        price_bundle["benchmark_code"],
        start_date=price_bundle["start_date"],
        end_date=price_bundle["end_date"],
        frequency="1d",
        fields="open",
        adjust_type=price_bundle["adjust_type"],
        skip_suspended=False,
        expect_df=True,
    )

    benchmark_open_panel = rqutils.to_wide_price_panel(
        benchmark_open_raw,
        value_column="open",
    )

    price_bundle["benchmark_open"] = (
        benchmark_open_panel.iloc[:, 0].copy()
    )

    pd.to_pickle(price_bundle, PRICE_CACHE_PATH)

    print("基准开盘价：已补充并更新本地价格缓存。")

else:
    print("基准开盘价：缓存命中。")

if BACKTEST_CACHE_PATH.exists():
    backtest_bundle = pd.read_pickle(BACKTEST_CACHE_PATH)

    daily_account = backtest_bundle["daily_account"]
    event_trade_summary = backtest_bundle["event_trade_summary"]
    performance_summary = backtest_bundle["performance_summary"]

    print("策略回测缓存：命中，直接读取。")

else:
    daily_account, event_trade_summary = (
        sc.run_open_price_rebalance_backtest(
            event_target_weights=sleeve_event_targets,
            open_prices=price_bundle["open_prices"],
            close_prices=price_bundle["close_prices"],
            initial_cash=INITIAL_CASH,
            buy_cost_rate=BUY_COST_RATE,
            sell_cost_rate=SELL_COST_RATE,
            min_commission=MIN_COMMISSION,
            lot_size=LOT_SIZE,
            cash_interest_annual=CASH_INTEREST_ANNUAL,
            annualization=ANNUALIZATION,
        )
    )

    benchmark_close = pd.Series(
        price_bundle["benchmark_close"]
    ).copy()

    benchmark_open = pd.Series(
        price_bundle["benchmark_open"]
    ).copy()

    benchmark_close.index = pd.to_datetime(
        benchmark_close.index
    ).normalize()

    benchmark_open.index = pd.to_datetime(
        benchmark_open.index
    ).normalize()

    benchmark_close = benchmark_close.reindex(
        daily_account.index
    ).ffill()

    benchmark_open = benchmark_open.reindex(
        daily_account.index
    ).ffill()

    first_backtest_date = daily_account.index[0]
    first_benchmark_open = benchmark_open.loc[first_backtest_date]

    if pd.isna(first_benchmark_open) or first_benchmark_open <= 0:
        raise RuntimeError("无法取得首个回测日的基准开盘价。")

    benchmark_nav = (
        INITIAL_CASH
        * benchmark_close
        / first_benchmark_open
    )

    performance_summary = sc.summarize_nav_performance(
        daily_account["nav"],
        initial_value=INITIAL_CASH,
        benchmark_nav=benchmark_nav,
        annualization=ANNUALIZATION,
        risk_free_annual=0.0,
    )

    backtest_bundle = {
        "daily_account": daily_account,
        "event_trade_summary": event_trade_summary,
        "performance_summary": performance_summary,
        "benchmark_nav": benchmark_nav,
    }

    BACKTEST_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(backtest_bundle, BACKTEST_CACHE_PATH)

    print("策略回测缓存：未命中，已完成并写入本地缓存。")

benchmark_nav = pd.Series(
    backtest_bundle["benchmark_nav"]
).reindex(daily_account.index)

daily_nav_export = daily_account.copy()
daily_nav_export["benchmark_nav"] = benchmark_nav
daily_nav_export["strategy_nav_normalized"] = (
    daily_nav_export["nav"] / INITIAL_CASH
)
daily_nav_export["benchmark_nav_normalized"] = (
    daily_nav_export["benchmark_nav"] / INITIAL_CASH
)

execution_summary = pd.DataFrame(
    [
        {
            "event_rebalance_count": len(event_trade_summary),
            "total_buy_notional": event_trade_summary[
                "buy_notional"
            ].sum(),
            "total_sell_notional": event_trade_summary[
                "sell_notional"
            ].sum(),
            "total_transaction_cost": event_trade_summary[
                "transaction_cost"
            ].sum(),
            "transaction_cost_pct_initial_cash": (
                event_trade_summary["transaction_cost"].sum()
                / INITIAL_CASH
            ),
            "total_unfilled_buy_notional": event_trade_summary[
                "unfilled_buy_notional"
            ].sum(),
            "open_mark_fallback_count": daily_account[
                "open_mark_fallback_count"
            ].sum(),
            "close_mark_fallback_count": daily_account[
                "close_mark_fallback_count"
            ].sum(),
            "mean_gross_market_exposure": daily_account[
                "gross_market_exposure"
            ].mean(),
        }
    ]
)

daily_nav_export.reset_index().to_csv(
    DAILY_NAV_PATH,
    index=False,
    encoding="utf-8-sig",
)

event_trade_summary.reset_index().to_csv(
    EVENT_TRADE_PATH,
    index=False,
    encoding="utf-8-sig",
)

performance_summary.to_csv(
    PERFORMANCE_PATH,
    index=False,
    encoding="utf-8-sig",
)

execution_summary.to_csv(
    EXECUTION_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("\n===== 策略绩效摘要 =====")
print(performance_summary.round(6).to_string(index=False))

print("\n===== 策略执行摘要 =====")
print(execution_summary.round(6).to_string(index=False))

print("\n本地回测缓存（不导出）：", BACKTEST_CACHE_PATH)
print("安全逐日净值：", DAILY_NAV_PATH)
print("安全事件交易汇总：", EVENT_TRADE_PATH)
print("安全策略绩效：", PERFORMANCE_PATH)
print("安全执行摘要：", EXECUTION_PATH)

print(
    "\n说明：这是含手续费、最低佣金、现金利息与逐日估值的开盘价策略模拟；"
    "价格涨跌停导致的委托拒绝尚未模拟，后续单独做执行约束敏感性检验。"
)

In [ ]:
# 06.6 冻结规则下的全样本、样本内、严格样本外策略绩效拆分
# 不重新计算因子、权重或价格，只读取 05/06 的本地缓存。

required_session_variables = [
    "PROJECT_ROOT",
    "pd",
    "plt",
    "sc",
]

missing_session_variables = [
    name for name in required_session_variables
    if name not in globals()
]

if missing_session_variables:
    raise RuntimeError(
        "请先更新并运行 06.0。缺少变量："
        + ", ".join(missing_session_variables)
    )

INITIAL_CASH = 100_000_000.0
BUY_COST_RATE = 0.0003
SELL_COST_RATE = 0.0008
MIN_COMMISSION = 5.0
LOT_SIZE = 100
CASH_INTEREST_ANNUAL = 0.02
ANNUALIZATION = 252

SCHEDULE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_eligible_schedule.csv"
)

Q5_WEIGHT_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_q5_target_weights.pkl"
)

PRICE_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_strategy_daily_prices.pkl"
)

FULL_BACKTEST_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_open_price_backtest.pkl"
)

SPLIT_BACKTEST_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_period_split_backtests.pkl"
)

PERIOD_PERFORMANCE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_period_performance.csv"
)

PERIOD_EXECUTION_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_period_execution.csv"
)

PERIOD_NAV_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_period_daily_nav.csv"
)

PERIOD_FIGURE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "figures"
    / "06_integrated_profitability_strategy_period_comparison.png"
)

required_files = [
    SCHEDULE_PATH,
    Q5_WEIGHT_CACHE_PATH,
    PRICE_CACHE_PATH,
    FULL_BACKTEST_CACHE_PATH,
]

missing_files = [
    str(path) for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "缺少必要的本地缓存或审计文件：\n"
        + "\n".join(missing_files)
    )

strategy_schedule = pd.read_csv(SCHEDULE_PATH)
strategy_schedule["signal_date"] = pd.to_datetime(
    strategy_schedule["signal_date"]
)
strategy_schedule["entry_date_t_plus_1_open"] = pd.to_datetime(
    strategy_schedule["entry_date_t_plus_1_open"]
)
strategy_schedule["exit_date_t_plus_21_open"] = pd.to_datetime(
    strategy_schedule["exit_date_t_plus_21_open"]
)

q5_target_weights = pd.read_pickle(Q5_WEIGHT_CACHE_PATH)
price_bundle = pd.read_pickle(PRICE_CACHE_PATH)
full_backtest_bundle = pd.read_pickle(FULL_BACKTEST_CACHE_PATH)

if "benchmark_open" not in price_bundle:
    raise RuntimeError(
        "价格缓存中缺少 benchmark_open，请先运行成功的 06.5。"
    )

open_prices = price_bundle["open_prices"]
close_prices = price_bundle["close_prices"]

benchmark_close = pd.Series(
    price_bundle["benchmark_close"]
).copy()

benchmark_open = pd.Series(
    price_bundle["benchmark_open"]
).copy()

benchmark_close.index = pd.to_datetime(
    benchmark_close.index
).normalize()

benchmark_open.index = pd.to_datetime(
    benchmark_open.index
).normalize()

period_definitions = [
    {
        "period": "in_sample_available_2019_2022",
        "signal_start": "2019-01-01",
        "signal_end": "2022-12-31",
    },
    {
        "period": "out_of_sample_signal_2023_2025",
        "signal_start": "2023-01-01",
        "signal_end": "2025-12-31",
    },
]

if SPLIT_BACKTEST_CACHE_PATH.exists():
    split_backtest_bundle = pd.read_pickle(
        SPLIT_BACKTEST_CACHE_PATH
    )
    print("分期策略回测缓存：命中，直接读取。")

else:
    split_backtest_bundle = {}

    for definition in period_definitions:
        period_name = definition["period"]

        period_schedule = strategy_schedule.loc[
            strategy_schedule["signal_date"].between(
                definition["signal_start"],
                definition["signal_end"],
            )
        ].copy()

        if period_schedule.empty:
            raise RuntimeError(
                f"{period_name} 没有可交易的冻结信号。"
            )

        period_event_targets, period_sleeve_audit = (
            sc.build_active_sleeve_event_targets(
                sleeve_weights=q5_target_weights,
                schedule=period_schedule,
                entry_column="entry_date_t_plus_1_open",
                exit_column="exit_date_t_plus_21_open",
                trading_calendar=open_prices.index,
            )
        )

        period_start = period_event_targets.index.min()
        period_end = period_event_targets.index.max()

        period_open_prices = open_prices.loc[
            period_start:period_end
        ]

        period_close_prices = close_prices.loc[
            period_start:period_end
        ]

        period_daily_account, period_event_trade_summary = (
            sc.run_open_price_rebalance_backtest(
                event_target_weights=period_event_targets,
                open_prices=period_open_prices,
                close_prices=period_close_prices,
                initial_cash=INITIAL_CASH,
                buy_cost_rate=BUY_COST_RATE,
                sell_cost_rate=SELL_COST_RATE,
                min_commission=MIN_COMMISSION,
                lot_size=LOT_SIZE,
                cash_interest_annual=CASH_INTEREST_ANNUAL,
                annualization=ANNUALIZATION,
            )
        )

        period_benchmark_close = benchmark_close.reindex(
            period_daily_account.index
        ).ffill()

        period_benchmark_open = benchmark_open.reindex(
            period_daily_account.index
        ).ffill()

        first_period_date = period_daily_account.index[0]
        first_benchmark_open = period_benchmark_open.loc[
            first_period_date
        ]

        if pd.isna(first_benchmark_open) or first_benchmark_open <= 0:
            raise RuntimeError(
                f"{period_name} 缺少首日基准开盘价。"
            )

        period_benchmark_nav = (
            INITIAL_CASH
            * period_benchmark_close
            / first_benchmark_open
        )

        period_performance = sc.summarize_nav_performance(
            period_daily_account["nav"],
            initial_value=INITIAL_CASH,
            benchmark_nav=period_benchmark_nav,
            annualization=ANNUALIZATION,
            risk_free_annual=0.0,
        )

        split_backtest_bundle[period_name] = {
            "schedule": period_schedule,
            "sleeve_audit": period_sleeve_audit,
            "daily_account": period_daily_account,
            "event_trade_summary": period_event_trade_summary,
            "benchmark_nav": period_benchmark_nav,
            "performance": period_performance,
        }

    SPLIT_BACKTEST_CACHE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.to_pickle(
        split_backtest_bundle,
        SPLIT_BACKTEST_CACHE_PATH,
    )

    print("分期策略回测缓存：未命中，已完成并写入本地缓存。")

full_schedule = strategy_schedule.copy()
full_daily_account = full_backtest_bundle["daily_account"]
full_event_trade_summary = full_backtest_bundle[
    "event_trade_summary"
]
full_benchmark_nav = pd.Series(
    full_backtest_bundle["benchmark_nav"]
)
full_performance = full_backtest_bundle["performance_summary"].copy()

full_performance.insert(
    0,
    "period",
    "full_available_2019_2025_signals",
)

full_performance["signal_period_count"] = len(full_schedule)
full_performance["first_signal_date"] = full_schedule[
    "signal_date"
].min()

full_performance["last_signal_date"] = full_schedule[
    "signal_date"
].max()

performance_frames = [full_performance]
execution_rows = []
nav_frames = []

for period_name, payload in {
    "full_available_2019_2025_signals": {
        "schedule": full_schedule,
        "daily_account": full_daily_account,
        "event_trade_summary": full_event_trade_summary,
        "benchmark_nav": full_benchmark_nav,
    },
    **split_backtest_bundle,
}.items():
    period_account = payload["daily_account"].copy()
    period_trade_summary = payload["event_trade_summary"].copy()
    period_benchmark_nav = pd.Series(
        payload["benchmark_nav"]
    ).reindex(period_account.index)

    if period_name != "full_available_2019_2025_signals":
        period_performance = payload["performance"].copy()
        period_performance.insert(0, "period", period_name)

        period_performance["signal_period_count"] = len(
            payload["schedule"]
        )

        period_performance["first_signal_date"] = payload[
            "schedule"
        ]["signal_date"].min()

        period_performance["last_signal_date"] = payload[
            "schedule"
        ]["signal_date"].max()

        performance_frames.append(period_performance)

    execution_rows.append(
        {
            "period": period_name,
            "signal_period_count": len(payload["schedule"]),
            "event_rebalance_count": len(period_trade_summary),
            "total_transaction_cost": period_trade_summary[
                "transaction_cost"
            ].sum(),
            "transaction_cost_pct_initial_cash": (
                period_trade_summary["transaction_cost"].sum()
                / INITIAL_CASH
            ),
            "total_unfilled_buy_notional": period_trade_summary[
                "unfilled_buy_notional"
            ].sum(),
            "mean_gross_market_exposure": period_account[
                "gross_market_exposure"
            ].mean(),
            "max_active_sleeves": (
                payload.get("sleeve_audit", pd.DataFrame(
                    {"active_sleeve_count": [2]}
                ))["active_sleeve_count"].max()
            ),
        }
    )

    period_nav_export = pd.DataFrame(
        {
            "date": period_account.index,
            "period": period_name,
            "strategy_nav": period_account["nav"].to_numpy(),
            "benchmark_nav": period_benchmark_nav.to_numpy(),
            "strategy_nav_normalized": (
                period_account["nav"].to_numpy() / INITIAL_CASH
            ),
            "benchmark_nav_normalized": (
                period_benchmark_nav.to_numpy() / INITIAL_CASH
            ),
        }
    )

    nav_frames.append(period_nav_export)

period_performance_summary = pd.concat(
    performance_frames,
    ignore_index=True,
)

period_execution_summary = pd.DataFrame(execution_rows)
period_nav_export = pd.concat(nav_frames, ignore_index=True)

period_performance_summary.to_csv(
    PERIOD_PERFORMANCE_PATH,
    index=False,
    encoding="utf-8-sig",
)

period_execution_summary.to_csv(
    PERIOD_EXECUTION_PATH,
    index=False,
    encoding="utf-8-sig",
)

period_nav_export.to_csv(
    PERIOD_NAV_PATH,
    index=False,
    encoding="utf-8-sig",
)

full_nav_plot = period_nav_export.loc[
    period_nav_export["period"].eq(
        "full_available_2019_2025_signals"
    )
].copy()

full_nav_plot["strategy_drawdown"] = (
    full_nav_plot["strategy_nav"]
    / full_nav_plot["strategy_nav"].cummax()
    - 1.0
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(
    full_nav_plot["date"],
    full_nav_plot["strategy_nav_normalized"],
    label="Strategy",
    linewidth=1.6,
)

axes[0].plot(
    full_nav_plot["date"],
    full_nav_plot["benchmark_nav_normalized"],
    label="CSI 800",
    linewidth=1.3,
)

axes[0].set_title("Integrated Profitability Strategy NAV")
axes[0].set_ylabel("Normalized NAV")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].fill_between(
    full_nav_plot["date"],
    full_nav_plot["strategy_drawdown"],
    0.0,
    color="firebrick",
    alpha=0.45,
)

axes[1].set_title("Strategy Drawdown")
axes[1].set_ylabel("Drawdown")
axes[1].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(PERIOD_FIGURE_PATH, dpi=180, bbox_inches="tight")
plt.show()

print("\n===== 分期策略绩效摘要 =====")
print(
    period_performance_summary[
        [
            "period",
            "signal_period_count",
            "total_return",
            "annualized_return",
            "benchmark_annualized_return",
            "annualized_excess_return",
            "annualized_volatility",
            "sharpe_ratio",
            "information_ratio",
            "max_drawdown",
        ]
    ].round(6).to_string(index=False)
)

print("\n===== 分期策略执行摘要 =====")
print(period_execution_summary.round(6).to_string(index=False))

print("\n已保存：", PERIOD_PERFORMANCE_PATH)
print("已保存：", PERIOD_EXECUTION_PATH)
print("已保存：", PERIOD_NAV_PATH)
print("已保存：", PERIOD_FIGURE_PATH)
print("本地缓存：", SPLIT_BACKTEST_CACHE_PATH)

In [ ]:
# 06.7 严格样本外策略归因：
# Q5 vs Q1 long control vs equal-weight eligible universe
# 三者共享相同信号、持有期、事件日开盘执行、交易成本与现金规则。

required_session_variables = [
    "PROJECT_ROOT",
    "pd",
    "plt",
    "rqdatac",
    "rqutils",
    "sc",
]

missing_session_variables = [
    name for name in required_session_variables
    if name not in globals()
]

if missing_session_variables:
    raise RuntimeError(
        "请先运行更新后的 06.0。缺少变量："
        + ", ".join(missing_session_variables)
    )

INITIAL_CASH = 100_000_000.0
BUY_COST_RATE = 0.0003
SELL_COST_RATE = 0.0008
MIN_COMMISSION = 5.0
LOT_SIZE = 100
CASH_INTEREST_ANNUAL = 0.02
ANNUALIZATION = 252

OOS_SIGNAL_START = "2023-01-01"
OOS_SIGNAL_END = "2025-12-31"
SELECTION_FRACTION = 0.20
MIN_VALID_STOCKS = 100

SCHEDULE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_strategy_eligible_schedule.csv"
)

FACTOR_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "05_integrated_profitability_structure_proxy.pkl"
)

Q5_WEIGHT_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_q5_target_weights.pkl"
)

ATTRIBUTION_TARGET_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_oos_attribution_targets.pkl"
)

ATTRIBUTION_PRICE_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_oos_attribution_prices.pkl"
)

ATTRIBUTION_BACKTEST_CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_oos_attribution_backtests.pkl"
)

SELECTION_AUDIT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_oos_attribution_selection_audit.csv"
)

PERFORMANCE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_oos_attribution_performance.csv"
)

RELATIVE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_oos_attribution_relative_performance.csv"
)

EXECUTION_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_oos_attribution_execution.csv"
)

NAV_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "tables"
    / "06_integrated_profitability_oos_attribution_daily_nav.csv"
)

FIGURE_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "figures"
    / "06_integrated_profitability_oos_attribution.png"
)

required_files = [
    SCHEDULE_PATH,
    FACTOR_CACHE_PATH,
    Q5_WEIGHT_CACHE_PATH,
]

missing_files = [
    str(path) for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "缺少必要文件：\n" + "\n".join(missing_files)
    )

strategy_schedule = pd.read_csv(SCHEDULE_PATH)

for column in [
    "signal_date",
    "entry_date_t_plus_1_open",
    "exit_date_t_plus_21_open",
]:
    strategy_schedule[column] = pd.to_datetime(
        strategy_schedule[column]
    )

oos_schedule = strategy_schedule.loc[
    strategy_schedule["signal_date"].between(
        OOS_SIGNAL_START,
        OOS_SIGNAL_END,
    )
].copy()

if len(oos_schedule) != 36:
    raise RuntimeError(
        "严格样本外应有 36 个可交易信号，"
        f"当前得到 {len(oos_schedule)} 个。"
    )

factor_cache_payload = pd.read_pickle(FACTOR_CACHE_PATH)

if (
    not isinstance(factor_cache_payload, dict)
    or "final_factor" not in factor_cache_payload
):
    raise RuntimeError(
        "综合因子缓存中找不到 final_factor。"
    )

final_factor_panel = factor_cache_payload["final_factor"].copy()
final_factor_panel.index = pd.to_datetime(
    final_factor_panel.index
).normalize()

q5_weights = pd.read_pickle(Q5_WEIGHT_CACHE_PATH)

q5_weights.index = pd.to_datetime(
    q5_weights.index
).normalize()

if ATTRIBUTION_TARGET_CACHE_PATH.exists():
    attribution_target_bundle = pd.read_pickle(
        ATTRIBUTION_TARGET_CACHE_PATH
    )

    q5_event_targets = attribution_target_bundle["q5_event_targets"]
    q1_event_targets = attribution_target_bundle["q1_event_targets"]
    universe_event_targets = attribution_target_bundle[
        "universe_event_targets"
    ]
    attribution_selection_audit = attribution_target_bundle[
        "selection_audit"
    ]

    print("归因目标权重缓存：命中，直接读取。")

else:
    q1_weights, q1_selection_audit = (
        sc.build_quantile_target_weights(
            score_panel=final_factor_panel,
            schedule=oos_schedule,
            signal_column="signal_date",
            entry_column="entry_date_t_plus_1_open",
            selection_fraction=SELECTION_FRACTION,
            min_valid_assets=MIN_VALID_STOCKS,
            higher_is_better=False,
        )
    )

    universe_weights, universe_selection_audit = (
        sc.build_quantile_target_weights(
            score_panel=final_factor_panel,
            schedule=oos_schedule,
            signal_column="signal_date",
            entry_column="entry_date_t_plus_1_open",
            selection_fraction=1.0,
            min_valid_assets=MIN_VALID_STOCKS,
            higher_is_better=True,
        )
    )

    q5_event_targets, q5_sleeve_audit = (
        sc.build_active_sleeve_event_targets(
            sleeve_weights=q5_weights,
            schedule=oos_schedule,
            entry_column="entry_date_t_plus_1_open",
            exit_column="exit_date_t_plus_21_open",
        )
    )

    q1_event_targets, q1_sleeve_audit = (
        sc.build_active_sleeve_event_targets(
            sleeve_weights=q1_weights,
            schedule=oos_schedule,
            entry_column="entry_date_t_plus_1_open",
            exit_column="exit_date_t_plus_21_open",
        )
    )

    universe_event_targets, universe_sleeve_audit = (
        sc.build_active_sleeve_event_targets(
            sleeve_weights=universe_weights,
            schedule=oos_schedule,
            entry_column="entry_date_t_plus_1_open",
            exit_column="exit_date_t_plus_21_open",
        )
    )

    q5_selection_audit = pd.DataFrame(
        {
            "signal_date": oos_schedule["signal_date"].to_numpy(),
            "entry_date": oos_schedule[
                "entry_date_t_plus_1_open"
            ].to_numpy(),
            "valid_score_count": (
                q5_weights.loc[
                    pd.to_datetime(
                        oos_schedule[
                            "entry_date_t_plus_1_open"
                        ]
                    ).dt.normalize()
                ]
                .notna()
                .sum(axis=1)
                .to_numpy()
            ),
            "selected_asset_count": (
                q5_weights.loc[
                    pd.to_datetime(
                        oos_schedule[
                            "entry_date_t_plus_1_open"
                        ]
                    ).dt.normalize()
                ]
                .ne(0.0)
                .sum(axis=1)
                .to_numpy()
            ),
            "eligible": True,
            "reason": "frozen_q5_strategy",
        }
    )

    q5_selection_audit["portfolio"] = "Q5"
    q1_selection_audit["portfolio"] = "Q1_long_control"
    universe_selection_audit["portfolio"] = (
        "equal_weight_eligible_universe"
    )

    attribution_selection_audit = pd.concat(
        [
            q5_selection_audit,
            q1_selection_audit,
            universe_selection_audit,
        ],
        ignore_index=True,
    )

    attribution_target_bundle = {
        "q5_event_targets": q5_event_targets,
        "q1_event_targets": q1_event_targets,
        "universe_event_targets": universe_event_targets,
        "selection_audit": attribution_selection_audit,
    }

    ATTRIBUTION_TARGET_CACHE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.to_pickle(
        attribution_target_bundle,
        ATTRIBUTION_TARGET_CACHE_PATH,
    )

    print("归因目标权重缓存：未命中，已新建。")

attribution_selection_audit.to_csv(
    SELECTION_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

portfolio_event_targets = {
    "Q5": q5_event_targets,
    "Q1_long_control": q1_event_targets,
    "equal_weight_eligible_universe": universe_event_targets,
}

attribution_asset_codes = sorted(
    {
        asset_code
        for target_panel in portfolio_event_targets.values()
        for asset_code in target_panel.columns[
            target_panel.ne(0.0).any(axis=0)
        ]
    }
)

attribution_start_date = min(
    target_panel.index.min()
    for target_panel in portfolio_event_targets.values()
).strftime("%Y-%m-%d")

attribution_end_date = max(
    target_panel.index.max()
    for target_panel in portfolio_event_targets.values()
).strftime("%Y-%m-%d")

price_cache_hit = False

if ATTRIBUTION_PRICE_CACHE_PATH.exists():
    cached_attribution_prices = pd.read_pickle(
        ATTRIBUTION_PRICE_CACHE_PATH
    )

    if (
        isinstance(cached_attribution_prices, dict)
        and {"open_prices", "close_prices"}.issubset(
            cached_attribution_prices
        )
    ):
        cached_open = cached_attribution_prices["open_prices"]

        cache_has_assets = set(attribution_asset_codes).issubset(
            set(cached_open.columns)
        )

        cache_has_dates = (
            pd.Timestamp(attribution_start_date)
            >= pd.to_datetime(cached_open.index).min()
            and pd.Timestamp(attribution_end_date)
            <= pd.to_datetime(cached_open.index).max()
        )

        if cache_has_assets and cache_has_dates:
            attribution_price_bundle = cached_attribution_prices
            price_cache_hit = True

if not price_cache_hit:
    print(
        "归因价格缓存：未命中，读取 "
        f"{len(attribution_asset_codes)} 只股票的开盘价与收盘价。"
    )

    attribution_open_raw = rqdatac.get_price(
        attribution_asset_codes,
        start_date=attribution_start_date,
        end_date=attribution_end_date,
        frequency="1d",
        fields="open",
        adjust_type="pre",
        skip_suspended=False,
        expect_df=True,
    )

    attribution_close_raw = rqdatac.get_price(
        attribution_asset_codes,
        start_date=attribution_start_date,
        end_date=attribution_end_date,
        frequency="1d",
        fields="close",
        adjust_type="pre",
        skip_suspended=False,
        expect_df=True,
    )

    attribution_price_bundle = {
        "open_prices": rqutils.to_wide_price_panel(
            attribution_open_raw,
            value_column="open",
        ).reindex(columns=attribution_asset_codes),
        "close_prices": rqutils.to_wide_price_panel(
            attribution_close_raw,
            value_column="close",
        ).reindex(columns=attribution_asset_codes),
        "asset_count": len(attribution_asset_codes),
        "start_date": attribution_start_date,
        "end_date": attribution_end_date,
    }

    ATTRIBUTION_PRICE_CACHE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.to_pickle(
        attribution_price_bundle,
        ATTRIBUTION_PRICE_CACHE_PATH,
    )

else:
    print("归因价格缓存：命中，直接读取。")

if ATTRIBUTION_BACKTEST_CACHE_PATH.exists():
    attribution_backtest_bundle = pd.read_pickle(
        ATTRIBUTION_BACKTEST_CACHE_PATH
    )

    print("归因策略回测缓存：命中，直接读取。")

else:
    attribution_backtest_bundle = {}

    for portfolio_name, target_panel in portfolio_event_targets.items():
        daily_account, event_trade_summary = (
            sc.run_open_price_rebalance_backtest(
                event_target_weights=target_panel,
                open_prices=attribution_price_bundle["open_prices"],
                close_prices=attribution_price_bundle["close_prices"],
                initial_cash=INITIAL_CASH,
                buy_cost_rate=BUY_COST_RATE,
                sell_cost_rate=SELL_COST_RATE,
                min_commission=MIN_COMMISSION,
                lot_size=LOT_SIZE,
                cash_interest_annual=CASH_INTEREST_ANNUAL,
                annualization=ANNUALIZATION,
            )
        )

        standalone_performance = sc.summarize_nav_performance(
            daily_account["nav"],
            initial_value=INITIAL_CASH,
            annualization=ANNUALIZATION,
            risk_free_annual=0.0,
        )

        attribution_backtest_bundle[portfolio_name] = {
            "daily_account": daily_account,
            "event_trade_summary": event_trade_summary,
            "performance": standalone_performance,
        }

    ATTRIBUTION_BACKTEST_CACHE_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.to_pickle(
        attribution_backtest_bundle,
        ATTRIBUTION_BACKTEST_CACHE_PATH,
    )

    print("归因策略回测缓存：未命中，已完成并写入本地缓存。")

performance_frames = []

for portfolio_name, payload in attribution_backtest_bundle.items():
    performance = payload["performance"].copy()
    performance.insert(0, "portfolio", portfolio_name)
    performance_frames.append(performance)

attribution_performance = pd.concat(
    performance_frames,
    ignore_index=True,
)

q5_nav = attribution_backtest_bundle["Q5"]["daily_account"]["nav"]
q1_nav = attribution_backtest_bundle[
    "Q1_long_control"
]["daily_account"]["nav"]

universe_nav = attribution_backtest_bundle[
    "equal_weight_eligible_universe"
]["daily_account"]["nav"]

q5_vs_q1 = sc.summarize_nav_performance(
    q5_nav,
    initial_value=INITIAL_CASH,
    benchmark_nav=q1_nav,
    annualization=ANNUALIZATION,
    risk_free_annual=0.0,
)

q5_vs_q1.insert(0, "comparison", "Q5_vs_Q1_long_control")

q5_vs_universe = sc.summarize_nav_performance(
    q5_nav,
    initial_value=INITIAL_CASH,
    benchmark_nav=universe_nav,
    annualization=ANNUALIZATION,
    risk_free_annual=0.0,
)

q5_vs_universe.insert(
    0,
    "comparison",
    "Q5_vs_equal_weight_eligible_universe",
)

relative_performance = pd.concat(
    [q5_vs_q1, q5_vs_universe],
    ignore_index=True,
)

execution_rows = []

for portfolio_name, payload in attribution_backtest_bundle.items():
    event_trade_summary = payload["event_trade_summary"]
    daily_account = payload["daily_account"]

    execution_rows.append(
        {
            "portfolio": portfolio_name,
            "event_rebalance_count": len(event_trade_summary),
            "total_transaction_cost": event_trade_summary[
                "transaction_cost"
            ].sum(),
            "transaction_cost_pct_initial_cash": (
                event_trade_summary["transaction_cost"].sum()
                / INITIAL_CASH
            ),
            "total_unfilled_buy_notional": event_trade_summary[
                "unfilled_buy_notional"
            ].sum(),
            "mean_gross_market_exposure": daily_account[
                "gross_market_exposure"
            ].mean(),
            "open_mark_fallback_count": daily_account[
                "open_mark_fallback_count"
            ].sum(),
            "close_mark_fallback_count": daily_account[
                "close_mark_fallback_count"
            ].sum(),
        }
    )

attribution_execution = pd.DataFrame(execution_rows)

nav_export_frames = []

for portfolio_name, payload in attribution_backtest_bundle.items():
    account = payload["daily_account"]

    nav_export_frames.append(
        pd.DataFrame(
            {
                "date": account.index,
                "portfolio": portfolio_name,
                "normalized_nav": (
                    account["nav"].to_numpy() / INITIAL_CASH
                ),
                "gross_market_exposure": account[
                    "gross_market_exposure"
                ].to_numpy(),
            }
        )
    )

attribution_nav = pd.concat(
    nav_export_frames,
    ignore_index=True,
)

attribution_performance.to_csv(
    PERFORMANCE_PATH,
    index=False,
    encoding="utf-8-sig",
)

relative_performance.to_csv(
    RELATIVE_PATH,
    index=False,
    encoding="utf-8-sig",
)

attribution_execution.to_csv(
    EXECUTION_PATH,
    index=False,
    encoding="utf-8-sig",
)

attribution_nav.to_csv(
    NAV_PATH,
    index=False,
    encoding="utf-8-sig",
)

plot_nav = attribution_nav.pivot(
    index="date",
    columns="portfolio",
    values="normalized_nav",
).sort_index()

relative_q5_q1 = (
    plot_nav["Q5"] / plot_nav["Q1_long_control"]
)

relative_q5_universe = (
    plot_nav["Q5"]
    / plot_nav["equal_weight_eligible_universe"]
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(
    plot_nav.index,
    plot_nav["Q5"],
    label="Q5",
    linewidth=1.7,
)

axes[0].plot(
    plot_nav.index,
    plot_nav["Q1_long_control"],
    label="Q1 long control",
    linewidth=1.2,
)

axes[0].plot(
    plot_nav.index,
    plot_nav["equal_weight_eligible_universe"],
    label="Equal-weight eligible universe",
    linewidth=1.2,
)

axes[0].set_title("Out-of-Sample Portfolio NAV Comparison")
axes[0].set_ylabel("Normalized NAV")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(
    relative_q5_q1.index,
    relative_q5_q1,
    label="Q5 / Q1",
    linewidth=1.5,
)

axes[1].plot(
    relative_q5_universe.index,
    relative_q5_universe,
    label="Q5 / Equal-weight universe",
    linewidth=1.5,
)

axes[1].axhline(1.0, color="black", linestyle="--", linewidth=0.8)
axes[1].set_title("Relative NAV of Q5")
axes[1].set_ylabel("Relative NAV")
axes[1].grid(alpha=0.25)
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGURE_PATH, dpi=180, bbox_inches="tight")
plt.show()

print("\n===== 样本外归因组合绩效 =====")
print(
    attribution_performance[
        [
            "portfolio",
            "total_return",
            "annualized_return",
            "annualized_volatility",
            "sharpe_ratio",
            "max_drawdown",
        ]
    ].round(6).to_string(index=False)
)

print("\n===== Q5 相对归因绩效 =====")
print(
    relative_performance[
        [
            "comparison",
            "annualized_excess_return",
            "information_ratio",
            "benchmark_annualized_return",
        ]
    ].round(6).to_string(index=False)
)

print("\n===== 样本外归因执行摘要 =====")
print(attribution_execution.round(6).to_string(index=False))

print("\n已保存：", SELECTION_AUDIT_PATH)
print("已保存：", PERFORMANCE_PATH)
print("已保存：", RELATIVE_PATH)
print("已保存：", EXECUTION_PATH)
print("已保存：", NAV_PATH)
print("已保存：", FIGURE_PATH)

print("本地目标权重缓存：", ATTRIBUTION_TARGET_CACHE_PATH)
print("本地价格缓存：", ATTRIBUTION_PRICE_CACHE_PATH)
print("本地回测缓存：", ATTRIBUTION_BACKTEST_CACHE_PATH)

In [ ]:
# 06.9 Python 3.9 中运行：
# 为 RQAlpha 的 Python3 环境导出可移植的冻结 Q5 事件日权重。

from pathlib import Path
import hashlib
import json
import pandas as pd

PROJECT_ROOT = Path(
    "/home/rice/notebook/a-share-profitability-factor-replication"
)

SOURCE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_oos_attribution_targets.pkl"
)

TARGET_PATH = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "07_rqalpha_oos_q5_event_targets.csv"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "cache"
    / "metadata"
    / "07_rqalpha_oos_q5_event_targets.json"
)

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"找不到源缓存：{SOURCE_PATH}")

source_bundle = pd.read_pickle(SOURCE_PATH)

if "q5_event_targets" not in source_bundle:
    raise RuntimeError("源缓存缺少 q5_event_targets。")

portable_targets = source_bundle["q5_event_targets"].copy()

portable_targets.index = pd.to_datetime(
    portable_targets.index
).normalize()

portable_targets = portable_targets.loc[
    :,
    portable_targets.ne(0.0).any(axis=0),
].fillna(0.0)

portable_targets.index.name = "event_date"

TARGET_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

portable_targets.reset_index().to_csv(
    TARGET_PATH,
    index=False,
    encoding="utf-8",
    float_format="%.16g",
)

file_sha256 = hashlib.sha256(
    TARGET_PATH.read_bytes()
).hexdigest()

manifest = {
    "purpose": "frozen_oos_q5_event_target_weights_for_rqalpha",
    "source_runtime": "Python3.9",
    "target_runtime": "RiceQuant Python3 RQAlpha environment",
    "source_cache": str(SOURCE_PATH),
    "target_csv": str(TARGET_PATH),
    "sha256": file_sha256,
    "event_date_count": int(len(portable_targets)),
    "asset_count": int(len(portable_targets.columns)),
    "first_event_date": portable_targets.index.min().strftime("%Y-%m-%d"),
    "last_event_date": portable_targets.index.max().strftime("%Y-%m-%d"),
    "minimum_target_gross_weight": float(
        portable_targets.sum(axis=1).min()
    ),
    "maximum_target_gross_weight": float(
        portable_targets.sum(axis=1).max()
    ),
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("===== RQAlpha 冻结权重导出完成 =====")
print("事件日数：", manifest["event_date_count"])
print("曾入选股票数：", manifest["asset_count"])
print("事件日期：", manifest["first_event_date"],
      "至", manifest["last_event_date"])
print("SHA256：", manifest["sha256"])
print("本地 CSV：", TARGET_PATH)
print("本地校验清单：", MANIFEST_PATH)

In [ ]:
# 仅一次：生成 Python 3.6 / RQAlpha 兼容的冻结 Q5 权重缓存
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")

SOURCE_CACHE = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_oos_attribution_targets.pkl"
)
RQALPHA_TARGET_CACHE = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "07_rqalpha_oos_q5_target_weights_py36.pkl"
)

source_bundle = pd.read_pickle(SOURCE_CACHE)
q5_targets = source_bundle["q5_event_targets"].copy()

q5_targets.to_pickle(RQALPHA_TARGET_CACHE, protocol=4)

print("已生成 RQAlpha 兼容权重缓存：", RQALPHA_TARGET_CACHE)
print("权重面板形状：", q5_targets.shape)

In [ ]:
from pathlib import Path
import pickle
import pandas as pd

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")

SOURCE_FILE = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_oos_attribution_targets.pkl"
)

RQALPHA_INPUT_DIR = PROJECT_ROOT / "data" / "strategy_inputs"
RQALPHA_INPUT_FILE = (
    RQALPHA_INPUT_DIR
    / "07_oos_q5_target_weights_py36.pkl"
)

RQALPHA_INPUT_DIR.mkdir(parents=True, exist_ok=True)

if RQALPHA_INPUT_FILE.exists():
    q5_targets = pd.read_pickle(RQALPHA_INPUT_FILE)
    print("已存在，直接读取 RQAlpha 输入权重文件。")
else:
    print("步骤 1/3：读取已冻结的因子研究权重。")
    source_bundle = pd.read_pickle(SOURCE_FILE)

    print("步骤 2/3：提取样本外 Q5 目标权重。")
    q5_targets = source_bundle["q5_event_targets"].copy()

    print("步骤 3/3：写入 Python 3.6 兼容的直接输入 pkl。")
    with RQALPHA_INPUT_FILE.open("wb") as file:
        pickle.dump(q5_targets, file, protocol=4)

print("RQAlpha 输入文件：", RQALPHA_INPUT_FILE)
print("权重面板形状：", q5_targets.shape)

In [ ]:
# 生成跨 Python / pandas 版本可读取的 RQAlpha 输入 pkl
from pathlib import Path
import pickle
import pandas as pd

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")

SOURCE_FILE = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_oos_attribution_targets.pkl"
)

PORTABLE_TARGET_FILE = (
    PROJECT_ROOT
    / "data"
    / "strategy_inputs"
    / "07_oos_q5_target_weights_portable.pkl"
)

PORTABLE_TARGET_FILE.parent.mkdir(parents=True, exist_ok=True)

print("步骤 1/3：读取冻结的研究权重。")
source_bundle = pd.read_pickle(SOURCE_FILE)
q5_targets = source_bundle["q5_event_targets"].copy()

print("步骤 2/3：转换为跨版本纯 Python 数据结构。")
portable_payload = {
    "format": "rqalpha_target_weights_portable_v1",
    "dates": pd.to_datetime(q5_targets.index).strftime("%Y-%m-%d").tolist(),
    "assets": q5_targets.columns.astype(str).tolist(),
    "weights": q5_targets.fillna(0.0).astype(float).values.tolist(),
}

print("步骤 3/3：写入 RQAlpha 直接输入 pkl。")
with PORTABLE_TARGET_FILE.open("wb") as file:
    pickle.dump(portable_payload, file, protocol=4)

print("已生成：", PORTABLE_TARGET_FILE)
print("日期数：", len(portable_payload["dates"]))
print("股票数：", len(portable_payload["assets"]))

In [ ]:
import sys
print(sys.version)

In [ ]:
# 07.4.0：导出 Q1 与等权可投资券池的跨环境 RQAlpha 权重输入
from pathlib import Path
import pickle

import pandas as pd

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")

SOURCE_FILE = (
    PROJECT_ROOT
    / "cache"
    / "data"
    / "06_integrated_profitability_oos_attribution_targets.pkl"
)

INPUT_DIR = PROJECT_ROOT / "data" / "strategy_inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

source_bundle = pd.read_pickle(SOURCE_FILE)

required_keys = {
    "q1_long_control": "q1_event_targets",
    "equal_weight_eligible_universe": "universe_event_targets",
}

missing_keys = [
    source_key
    for source_key in required_keys.values()
    if source_key not in source_bundle
]

if missing_keys:
    raise RuntimeError(
        "冻结的归因权重缓存字段与预期不一致。\n"
        "缺少：" + ", ".join(missing_keys) + "\n"
        "实际字段：" + ", ".join(sorted(source_bundle.keys()))
    )

for portfolio_name, source_key in required_keys.items():
    target_weights = source_bundle[source_key].copy()
    target_weights.index = pd.to_datetime(
        target_weights.index
    ).normalize()

    portable_payload = {
        "format": "rqalpha_target_weights_portable_v1",
        "portfolio_name": portfolio_name,
        "dates": target_weights.index.strftime("%Y-%m-%d").tolist(),
        "assets": target_weights.columns.astype(str).tolist(),
        "weights": (
            target_weights.fillna(0.0)
            .astype(float)
            .values
            .tolist()
        ),
    }

    output_file = (
        INPUT_DIR
        / "07_{}_target_weights_portable.pkl".format(portfolio_name)
    )

    with output_file.open("wb") as file:
        pickle.dump(portable_payload, file, protocol=4)

    print("已生成：", output_file)
    print(
        "  日期数：{}；股票列数：{}；日期范围：{} 至 {}".format(
            len(portable_payload["dates"]),
            len(portable_payload["assets"]),
            portable_payload["dates"][0],
            portable_payload["dates"][-1],
        )
    )